# 09 - Model: TF-IDF

# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/processed/featured_dataset.csv')

with open('../config/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

print('shape:', df.shape)
print('sample tfidf_input:')
print(df['tfidf_input'].head(5).tolist())

shape: (88167, 32)
sample tfidf_input:
['acoustic j-pop singer-songwriter songwriter calm slow mainstream', 'acoustic chill sad slow mainstream', 'acoustic sad slow mainstream', 'acoustic sad fast mainstream', 'acoustic sad medium charttoppers']


# TF-IDF Vectorizer

In [2]:
# fit tfidf vectorizer
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['tfidf_input'])

print('tfidf matrix shape:', tfidf_matrix.shape)
print('vocabulary size:', len(tfidf.vocabulary_))
print('\nsample vocabulary:')
print(list(tfidf.vocabulary_.keys())[:20])

tfidf matrix shape: (88167, 123)
vocabulary size: 123

sample vocabulary:
['acoustic', 'pop', 'singer', 'songwriter', 'calm', 'slow', 'mainstream', 'chill', 'sad', 'fast', 'medium', 'charttoppers', 'indie', 'piano', 'rock', 'angry', 'guitar', 'emerging', 'happy', 'folk']


# TF-IDF Recommendation Function

In [3]:
def recommend_tfidf(genre, df, tfidf_matrix, tfidf, n=10):
    # create query from genre
    query = genre.lower().replace(',', ' ')
    print(f'Query: "{query}"')
    print()
    
    # transform query
    query_vector = tfidf.transform([query])
    
    # compute similarity
    similarity = cosine_similarity(query_vector, tfidf_matrix)[0]
    
    # get top n
    top_indices = similarity.argsort()[::-1][:n]
    
    results = df.iloc[top_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity'] = similarity[top_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# Experiment 1 - Single Genre Query

In [4]:
for genre in ['rock', 'pop', 'jazz', 'hip-hop', 'classical']:
    print(f'\n{"="*40}')
    print(f'Genre: {genre}')
    print('='*40)
    results = recommend_tfidf(genre, df, tfidf_matrix, tfidf)
    print(results[['track_name', 'artists', 'track_genre', 'popularity', 'similarity']])


Genre: rock
Query: "rock"

                       track_name            artists        track_genre  \
1                            雨あがり             WANIMA  j-rock, punk-rock   
2   Psycho Killer - 2005 Remaster      Talking Heads    punk-rock, rock   
3                           サウダージ     PornoGraffitti             j-rock   
4                            No.1  Noriyuki Makihara             j-rock   
5                         BE FREE            GReeeeN             j-rock   
6                         SINGLES        Mr.Children             j-rock   
7                   せつなさを殺せない2014       Koji Kikkawa             j-rock   
8                             Piç               Peyk             j-rock   
9     Te Amar Me Faz Feliz Demais            Onze:20             j-rock   
10                       Bul Beni        Yavuz Çetin             j-rock   

    popularity  similarity  
1           34      0.7698  
2            0      0.7697  
3           33      0.7569  
4           34      0.7569  
5

# Improved TF-IDF with Popularity and Deduplication

In [5]:
def recommend_tfidf_v2(genre, df, tfidf_matrix, tfidf, n=10):
    query = genre.lower().replace(',', ' ')
    print(f'Query: "{query}"')
    print()
    
    query_vector = tfidf.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix)[0]
    
    df_temp = df.copy()
    df_temp['similarity'] = similarity
    df_temp['popularity_norm'] = df_temp['popularity'] / 100
    df_temp['combined_score'] = (
        0.6 * df_temp['similarity'] + 
        0.4 * df_temp['popularity_norm']
    )
    
    # filter only songs containing the genre
    genre_mask = df_temp['track_genre'].str.contains(genre.lower(), case=False)
    df_filtered = df_temp[genre_mask]
    
    # deduplicate by track name
    df_filtered = df_filtered.drop_duplicates(subset=['track_name'])
    
    results = df_filtered.nlargest(n, 'combined_score')[
        ['track_name', 'artists', 'track_genre', 'popularity', 
         'similarity', 'combined_score']
    ].copy()
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

for genre in ['rock', 'pop', 'jazz', 'hip-hop', 'classical']:
    print(f'\n{"="*40}')
    print(f'Genre: {genre}')
    print('='*40)
    results = recommend_tfidf_v2(genre, df, tfidf_matrix, tfidf)
    print(results[['track_name', 'artists', 'track_genre', 'popularity']])


Genre: rock
Query: "rock"

                                           track_name  \
1                                     Highway to Hell   
2                                                 紅蓮華   
3                                       Thunderstruck   
4                                       Back In Black   
5                                Don't Stop Believin'   
6                                    Comfortably Numb   
7   Separate Ways (Worlds Apart) - Bryce Miller/Al...   
8                                     Mary On A Cross   
9                          The Man Who Can't Be Moved   
10                                   Everything Black   

                     artists             track_genre  popularity  
1                      AC/DC         hard-rock, rock          85  
2                       LiSA     anime, j-rock, rock          74  
3                      AC/DC         hard-rock, rock          84  
4                      AC/DC         hard-rock, rock          85  
5        

# Experiment 3 - Specific Genre Queries

In [6]:
specific_queries = ['western pop', 'american hip-hop', 'rock angry', 'jazz calm', 'classical sad']

for query in specific_queries:
    print(f'\n{"="*40}')
    print(f'Query: {query}')
    print('='*40)
    
    # use first word as genre filter
    genre_filter = query.split()[0] if query.split()[0] in df['track_genre'].str.cat(sep=' ') else query.split()[-1]
    
    query_vector = tfidf.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix)[0]
    
    df_temp = df.copy()
    df_temp['similarity'] = similarity
    df_temp['popularity_norm'] = df_temp['popularity'] / 100
    df_temp['combined_score'] = 0.6 * df_temp['similarity'] + 0.4 * df_temp['popularity_norm']
    
    results = df_temp.drop_duplicates(subset=['track_name']).nlargest(5, 'combined_score')[
        ['track_name', 'artists', 'track_genre', 'popularity', 'combined_score']
    ]
    results = results.reset_index(drop=True)
    results.index += 1
    print(results)


Query: western pop
                             track_name                        artists  \
1                     How You Like That                      BLACKPINK   
2                                  SOLO                         JENNIE   
3  Kanja Poovu Kannala (From "Viruman")  Yuvan Shankar Raja;Sid Sriram   
4                            Pink Venom                      BLACKPINK   
5                     Agar Tum Saath Ho       Alka Yagnik;Arijit Singh   

            track_genre  popularity  combined_score  
1            k-pop, pop          75        0.835717  
2            k-pop, pop          73        0.827717  
3  k-pop, pop, pop-film          74        0.809394  
4            k-pop, pop          85        0.807800  
5  k-pop, pop, pop-film          73        0.804574  

Query: american hip-hop
                                          track_name  \
1              Quevedo: Bzrp Music Sessions, Vol. 52   
2                                               OOPS   
3  Dippam Dappam (

# Experiment 4 - TF-IDF Parameters

In [7]:
configs = {
    'default': TfidfVectorizer(),
    'bigrams': TfidfVectorizer(ngram_range=(1, 2)),
    'no_sublinear': TfidfVectorizer(sublinear_tf=False),
    'sublinear': TfidfVectorizer(sublinear_tf=True),
}

for name, vectorizer in configs.items():
    matrix = vectorizer.fit_transform(df['tfidf_input'])
    query_vec = vectorizer.transform(['rock'])
    sim = cosine_similarity(query_vec, matrix)[0]
    
    df_temp = df.copy()
    df_temp['similarity'] = sim
    df_temp['combined_score'] = 0.6 * sim + 0.4 * df_temp['popularity'] / 100
    
    genre_mask = df_temp['track_genre'].str.contains('rock', case=False)
    results = df_temp[genre_mask].drop_duplicates(subset=['track_name']).nlargest(5, 'combined_score')
    
    print(f'\n{"="*40}')
    print(f'Config: {name} | vocab size: {len(vectorizer.vocabulary_)}')
    print('='*40)
    print(results[['track_name', 'artists', 'popularity', 'combined_score']].to_string())


Config: default | vocab size: 123
                 track_name  artists  popularity  combined_score
41104       Highway to Hell    AC/DC          85        0.744173
4339                    紅蓮華     LiSA          74        0.741533
41099         Thunderstruck    AC/DC          84        0.740173
41107         Back In Black    AC/DC          85        0.739626
41132  Don't Stop Believin'  Journey          83        0.737057

Config: bigrams | vocab size: 1246
                 track_name  artists  popularity  combined_score
41104       Highway to Hell    AC/DC          85        0.608124
41099         Thunderstruck    AC/DC          84        0.604124
41132  Don't Stop Believin'  Journey          83        0.603565
41107         Back In Black    AC/DC          85        0.600841
41094       Mary On A Cross    Ghost          88        0.591426

Config: no_sublinear | vocab size: 123
                 track_name  artists  popularity  combined_score
41104       Highway to Hell    AC/DC        

# Save Best Model

In [8]:
import pickle

# refit best tfidf
best_tfidf = TfidfVectorizer()
best_matrix = best_tfidf.fit_transform(df['tfidf_input'])

best_config = {
    'model': 'tfidf_v2',
    'vectorizer': 'TfidfVectorizer default',
    'query_type': 'genre + mood + tempo',
    'combined_score': {
        'similarity_weight': 0.6,
        'popularity_weight': 0.4
    },
    'deduplication': True,
    'genre_filter': True
}

os.makedirs('../models/content_based', exist_ok=True)

# save tfidf matrix
import scipy.sparse as sp
sp.save_npz('../models/content_based/tfidf_matrix.npz', best_matrix)

# save vectorizer
with open('../models/content_based/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(best_tfidf, f)

# save config
with open('../models/content_based/tfidf_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

# save song index
df[['track_id', 'track_name', 'artists', 'track_genre', 
    'popularity', 'tfidf_input']].to_csv(
    '../models/content_based/tfidf_song_index.csv', index=True)

print('saved files:')
print('1. models/content_based/tfidf_matrix.npz')
print('2. models/content_based/tfidf_vectorizer.pkl')
print('3. models/content_based/tfidf_config.json')
print('4. models/content_based/tfidf_song_index.csv')

saved files:
1. models/content_based/tfidf_matrix.npz
2. models/content_based/tfidf_vectorizer.pkl
3. models/content_based/tfidf_config.json
4. models/content_based/tfidf_song_index.csv


# Conclusion

In [9]:
print('='*50)
print('TF-IDF MODEL - CONCLUSION')
print('='*50)

print('\n--- Experiments Summary ---')
print('Exp 1: basic tfidf (no filter)          → failed: duplicates, low popularity')
print('Exp 2: tfidf v2 (filter + popularity)   → best: clean, relevant, popular')
print('Exp 3: genre + mood queries             → excellent: mood+genre combinations')
print('Exp 4: tfidf parameters                 → default config is best')

print('\n--- Best Configuration ---')
print('Vectorizer     → TfidfVectorizer (default)')
print('Combined score → 60% similarity + 40% popularity')
print('Genre filter   → exact genre match')
print('Deduplication  → by track name')

print('\n--- Key Findings ---')
print('1. basic tfidf without filter gives poor results')
print('2. genre filter essential for relevant recommendations')
print('3. popularity weighting fixes low popularity bias')
print('4. deduplication removes repeated songs')
print('5. genre + mood queries work excellently')
print('6. bigrams add complexity without clear benefit')
print('7. sublinear tf gives more diverse but less popular results')
print('8. vocabulary of 123 words is sufficient')

print('\n--- Best Query Types ---')
print('single genre  → "rock", "jazz", "classical"')
print('genre + mood  → "rock angry", "jazz calm", "classical sad"')
print('genre + tempo → "rock fast", "jazz slow"')

print('\n--- Limitations ---')
print('1. unknown words ignored (western, american)')
print('2. k-pop dominates pop queries (multiple pop-related words)')
print('3. regional music can dominate niche genre queries')

print('\n--- Best Use Case ---')
print('genre based recommendation')
print('works well for: recommend rock/jazz/classical songs')
print('bonus: genre + mood combination queries')

print('\n--- Saved Files ---')
print('models/content_based/tfidf_matrix.npz')
print('models/content_based/tfidf_vectorizer.pkl')
print('models/content_based/tfidf_config.json')
print('models/content_based/tfidf_song_index.csv')

TF-IDF MODEL - CONCLUSION

--- Experiments Summary ---
Exp 1: basic tfidf (no filter)          → failed: duplicates, low popularity
Exp 2: tfidf v2 (filter + popularity)   → best: clean, relevant, popular
Exp 3: genre + mood queries             → excellent: mood+genre combinations
Exp 4: tfidf parameters                 → default config is best

--- Best Configuration ---
Vectorizer     → TfidfVectorizer (default)
Combined score → 60% similarity + 40% popularity
Genre filter   → exact genre match
Deduplication  → by track name

--- Key Findings ---
1. basic tfidf without filter gives poor results
2. genre filter essential for relevant recommendations
3. popularity weighting fixes low popularity bias
4. deduplication removes repeated songs
5. genre + mood queries work excellently
6. bigrams add complexity without clear benefit
7. sublinear tf gives more diverse but less popular results
8. vocabulary of 123 words is sufficient

--- Best Query Types ---
single genre  → "rock", "jazz", "cl